# Introduction


We are parsing and processing Enron emails, then save a CSV file with the processed emails. We will use them for two objectives:
- to add them in a vector database and query then the content of the database using a Retrieval Augmented Generation system - created using Llama 2 and Langchain.  
- to create a graph visualization of the email correspondence inside the company.

# Data preparation

In [1]:
import os, sys, email
import numpy as np 
import pandas as pd

In [2]:
emails_df = pd.read_csv('/kaggle/input/enron-email-dataset/emails.csv')
print(emails_df.shape)
emails_df.head()

(517401, 2)


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [3]:
print(emails_df['message'][0])

Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


In [4]:
def get_text_from_email(msg):
    '''To get the content from email objects'''
    parts = []
    for part in msg.walk():
        if part.get_content_type() == 'text/plain':
            parts.append( part.get_payload() )
    return ''.join(parts)

def split_email_addresses(line):
    '''To separate multiple email addresses'''
    if line:
        addrs = line.split(',')
        addrs = frozenset(map(lambda x: x.strip(), addrs))
    else:
        addrs = None
    return addrs

In [5]:
# work with a small selection

s_email_df = emails_df.sample(100_000)

In [6]:
messages = list(map(email.message_from_string, s_email_df['message']))
s_email_df.drop('message', axis=1, inplace=True)
# Get fields from parsed email objects
keys = messages[0].keys()
for key in keys:
    s_email_df[key] = [doc[key] for doc in messages]
# Parse content from emails
s_email_df['content'] = list(map(get_text_from_email, messages))
# Split multiple email addresses
s_email_df['From'] = s_email_df['From'].map(split_email_addresses)
s_email_df['To'] = s_email_df['To'].map(split_email_addresses)

# Extract the root of 'file' as 'user'
s_email_df['user'] = s_email_df['file'].map(lambda x:x.split('/')[0])
del messages

s_email_df.head()

,file,Message-ID,Date,From,To,Subject,Mime-Version,Content-Type,Content-Transfer-Encoding,X-From,X-To,X-cc,X-bcc,X-Folder,X-Origin,X-FileName,content,user
13668,bass-e/deleted_items/354.,<31810966.1075861326559.JavaMail.evans@thyme>,"Mon, 26 Nov 2001 11:55:45 -0800 (PST)",(great_steaks@offer.omahasteaks.com),(ebass@enron.com),Free Cutlery Set and Cutting Board!,1.0,text/plain; charset=us-ascii,7bit,Omaha Steaks <Great_Steaks@offer.omahasteaks.com>,ebass@enron.com,,,"\EBASS (Non-Privileged)\Bass, Eric\Deleted Items",Bass-E,EBASS (Non-Privileged).pst,"Dear Eric,\n\nOmaha Steaks is cutting you a te...",bass-e
177085,hyvl-d/all_documents/741.,<20333856.1075842229187.JavaMail.evans@thyme>,"Thu, 21 Dec 2000 05:28:00 -0800 (PST)",(teresa.bushman@enron.com),(dan.hyvl@enron.com),Possible Gas contract,1.0,text/plain; charset=us-ascii,7bit,Teresa G Bushman,Dan J Hyvl,,,\Dan_Hyvl_Dec2000_June2001\Notes Folders\All d...,HYVL-D,dhyvl.nsf,\n\n\n\nTeresa G. Bushman\nEnron North America...,hyvl-d
425942,shackleton-s/policy/81.,<27529712.1075863629959.JavaMail.evans@thyme>,"Thu, 29 Jun 2000 04:46:00 -0700 (PDT)",(office.chairman@enron.com),(esop.a.@enron.com),EnronOptions - Your Stock Option Program,1.0,text/plain; charset=ANSI_X3.4-1968,quoted-printable,Office of the Chairman,ESOP All Enron N. A.,,,\Sara_Shackleton_Dec2000_June2001_1\Notes Fold...,SHACKLETON-S,sshackle.nsf,It is amazing and yet not surprising how much ...,shackleton-s
110723,farmer-d/sent/118.,<30279112.1075854152044.JavaMail.evans@thyme>,"Fri, 29 Sep 2000 06:35:00 -0700 (PDT)",(daren.farmer@enron.com),(robert.walker@enron.com),Re: Cornhusker Contact Information,1.0,text/plain; charset=us-ascii,7bit,Daren J Farmer,Robert Walker,,,\Darren_Farmer_Dec2000\Notes Folders\Sent,Farmer-D,dfarmer.nsf,ENA Contact\n\nDaren Farmer\nPhone # 713-853-6...,farmer-d
484607,taylor-m/sent/1437.,<10273095.1075860082231.JavaMail.evans@thyme>,"Thu, 26 Oct 2000 02:31:00 -0700 (PDT)",(mark.taylor@enron.com),(tana.jones@enron.com),Re: Tax Re: i2 Technologies,1.0,text/plain; charset=us-ascii,7bit,Mark Taylor,Tana Jones,,,\Mark_Taylor _Dec_2000\Notes Folders\Sent,Taylor-M,mtaylor.nsf,Agreed - can you please confirm directly with ...,taylor-m


In [7]:
# selection

to_selection = s_email_df['X-To'].value_counts()[0:100].index
from_selection = s_email_df['X-From'].value_counts()[0:100].index

sc_email_df = s_email_df.loc[s_email_df['X-From'].isin(to_selection) & s_email_df['X-From'].isin(from_selection)]
sc_email_df.shape

(24069, 18)

In [8]:
sc_email_df = sc_email_df[['To', 'From', 'X-To', 'X-From', 'content']]

In [9]:
sc_email_df.head(20)

,To,From,X-To,X-From,content
110723,(robert.walker@enron.com),(daren.farmer@enron.com),Robert Walker,Daren J Farmer,ENA Contact\n\nDaren Farmer\nPhone # 713-853-6...
484607,(tana.jones@enron.com),(mark.taylor@enron.com),Tana Jones,Mark Taylor,Agreed - can you please confirm directly with ...
227720,(elizabeth.linnell@enron.com),(steven.kean@enron.com),Elizabeth Linnell,Steven J Kean,Please send Pam Benson the information for Joe...
196929,(bob.shults@enron.com),(tana.jones@enron.com),Bob Shults,Tana Jones,Blue Range Resources Corporation is an execute...
244692,"(charles.yeung@enron.com, richard.ingersoll@en...",(steven.kean@enron.com),"Christi L Nicolay, Richard Ingersoll, Charles ...",Steven J Kean,----- Forwarded by Steven J Kean/NA/Enron on 0...
353547,(andrew.miles@enron.com),(gerald.nemec@enron.com),Andrew Miles,Gerald Nemec,"Andrew, Attached is the CSA. I added some la..."
321469,(ben.jacoby@enron.com),(kay.mann@enron.com),Ben F Jacoby,Kay Mann,Are you here? I wanted to get those Intergen ...
433822,(shanna.funkhouser@enron.com),(jeffrey.shankman@enron.com),Shanna Funkhouser,Jeffrey A Shankman,FYI. \n---------------------- Forwarded by Jef...
81007,"(harry.kingerski@enron.com, susan.mara@enron.com)",(jeff.dasovich@enron.com),"Harry Kingerski, Susan J Mara",Jeff Dasovich,----- Forwarded by Jeff Dasovich/NA/Enron on 0...
303098,(nmann@erac.com),(kay.mann@enron.com),<nmann@erac.com>,Kay Mann,YIPPEEE.\n\nI'm about to fall face down in my ...


In [10]:
m_email_df = sc_email_df.loc[~sc_email_df["X-To"].str.contains('@')]
m_email_df = m_email_df.loc[~m_email_df["X-From"].str.contains('@')]

m_email_df = m_email_df.loc[~m_email_df["X-From"].str.contains('<')]
m_email_df = m_email_df.loc[~m_email_df["X-To"].str.contains('<')]

m_email_df = m_email_df.loc[~m_email_df["X-To"].str.contains(',')]
m_email_df = m_email_df.loc[~m_email_df["To"].isna()]

In [11]:
m_email_df.shape

(13508, 5)

In [12]:
m_email_df.head()

,To,From,X-To,X-From,content
110723,(robert.walker@enron.com),(daren.farmer@enron.com),Robert Walker,Daren J Farmer,ENA Contact\n\nDaren Farmer\nPhone # 713-853-6...
484607,(tana.jones@enron.com),(mark.taylor@enron.com),Tana Jones,Mark Taylor,Agreed - can you please confirm directly with ...
227720,(elizabeth.linnell@enron.com),(steven.kean@enron.com),Elizabeth Linnell,Steven J Kean,Please send Pam Benson the information for Joe...
196929,(bob.shults@enron.com),(tana.jones@enron.com),Bob Shults,Tana Jones,Blue Range Resources Corporation is an execute...
353547,(andrew.miles@enron.com),(gerald.nemec@enron.com),Andrew Miles,Gerald Nemec,"Andrew, Attached is the CSA. I added some la..."


In [13]:
m_email_df["to_index"] = "From " + m_email_df["X-From"] + " to " + m_email_df["X-To"] + ": " + + m_email_df["content"]

In [14]:
m_email_df.head()

,To,From,X-To,X-From,content,to_index
110723,(robert.walker@enron.com),(daren.farmer@enron.com),Robert Walker,Daren J Farmer,ENA Contact\n\nDaren Farmer\nPhone # 713-853-6...,From Daren J Farmer to Robert Walker: ENA Cont...
484607,(tana.jones@enron.com),(mark.taylor@enron.com),Tana Jones,Mark Taylor,Agreed - can you please confirm directly with ...,From Mark Taylor to Tana Jones: Agreed - can y...
227720,(elizabeth.linnell@enron.com),(steven.kean@enron.com),Elizabeth Linnell,Steven J Kean,Please send Pam Benson the information for Joe...,From Steven J Kean to Elizabeth Linnell: Pleas...
196929,(bob.shults@enron.com),(tana.jones@enron.com),Bob Shults,Tana Jones,Blue Range Resources Corporation is an execute...,From Tana Jones to Bob Shults: Blue Range Reso...
353547,(andrew.miles@enron.com),(gerald.nemec@enron.com),Andrew Miles,Gerald Nemec,"Andrew, Attached is the CSA. I added some la...","From Gerald Nemec to Andrew Miles: Andrew, At..."


In [15]:
m_email_df.to_csv("proc_email.csv", index=False)